In [1]:
import copy
import sys
from pathlib import Path
from uuid import uuid4

import torch
import torch.nn as nn

sys.path.insert(0, str(Path("../src").resolve()))
from cnn_project.model import EuroSATModel
from cnn_project.data import create_data_loaders
from cnn_project.evaluate import evaluate_model
from cnn_project.train import train_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_root = Path("../Checkpoints")
runs = [p for p in checkpoint_root.glob("width_comparison_*")
        if all((p / f"eurosat_cnn_best_model{i}.pt").is_file() for i in (1, 2, 3))]
if not runs:
    raise FileNotFoundError("No complete width-comparison run found in ../Checkpoints")
source_run = max(runs, key=lambda p: p.stat().st_mtime)
print(f"Loading checkpoints from {source_run} on {device}")

model_widths = {
    "model1": (16, 32, 64, 128),
    "model2": (32, 64, 128, 256),
    "model3": (64, 128, 256, 512),
}
models = {}
for name, channels in model_widths.items():
    model = EuroSATModel(channels=channels).to(device)
    checkpoint = source_run / f"eurosat_cnn_best_{name}.pt"
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    model.eval()
    models[name] = model


Loading checkpoints from ../Checkpoints/width_comparison_495773002b8d44c78cce4383ad77f925 on cuda


In [2]:
def prune_filters(model, prune_ratio=0.35):
    """Remove approximately prune_ratio of output filters in each convolution."""
    if not 0 <= prune_ratio < 1:
        raise ValueError("prune_ratio must be between 0 (inclusive) and 1 (exclusive)")
    pruned = copy.deepcopy(model)
    layers = pruned.model
    previous_keep = torch.arange(3, device=next(model.parameters()).device)

    with torch.no_grad():
        for conv_idx in (0, 4, 8, 12):
            conv = layers[conv_idx]
            bn = layers[conv_idx + 1]
            scores = conv.weight.abs().sum(dim=(1, 2, 3))
            keep_count = max(1, round(conv.out_channels * (1 - prune_ratio)))
            keep = scores.topk(keep_count).indices.sort().values

            new_conv = nn.Conv2d(
                len(previous_keep), len(keep), kernel_size=conv.kernel_size,
                stride=conv.stride, padding=conv.padding, dilation=conv.dilation,
                bias=False,
            ).to(device=conv.weight.device, dtype=conv.weight.dtype)
            new_conv.weight.copy_(conv.weight[keep][:, previous_keep])

            new_bn = nn.BatchNorm2d(len(keep), eps=bn.eps, momentum=bn.momentum).to(
                device=bn.weight.device, dtype=bn.weight.dtype
            )
            new_bn.weight.copy_(bn.weight[keep])
            new_bn.bias.copy_(bn.bias[keep])
            new_bn.running_mean.copy_(bn.running_mean[keep])
            new_bn.running_var.copy_(bn.running_var[keep])
            new_bn.num_batches_tracked.copy_(bn.num_batches_tracked)
            layers[conv_idx] = new_conv
            layers[conv_idx + 1] = new_bn
            print(f"  Convolution {conv_idx}: {conv.out_channels} -> {len(keep)} filters")
            previous_keep = keep

        fc = layers[16]
        features_per_channel = 4 * 4
        columns = (previous_keep[:, None] * features_per_channel +
                   torch.arange(features_per_channel, device=fc.weight.device)).reshape(-1)
        new_fc = nn.Linear(len(columns), fc.out_features).to(
            device=fc.weight.device, dtype=fc.weight.dtype
        )
        new_fc.weight.copy_(fc.weight[:, columns])
        new_fc.bias.copy_(fc.bias)
        layers[16] = new_fc
    pruned.eval()
    return pruned


In [3]:
# Model 1 keeps the original 35% pruning. Ratios for models 2 and 3
# target approximately 0.4 MB of parameters and buffers (decimal MB).
prune_ratios = {"model1": 0.35, "model2": 0.543, "model3": 0.7715}
pruned_models = {}
for name, model in models.items():
    print(f"\n{name}: pruning {prune_ratios[name]:.2%} of filters per layer")
    pruned = prune_filters(model, prune_ratios[name])
    pruned_models[name] = pruned
    with torch.inference_mode():
        output = pruned(torch.randn(1, 3, 64, 64, device=device))
    before, before_mb = model.size()
    after, after_mb = pruned.size()
    print(f"  Output shape: {tuple(output.shape)}")
    print(f"  Parameters: {before:,} -> {after:,} ({100 * (1 - after / before):.1f}% reduction)")
    print(f"  Size: {before_mb:.4f} -> {after_mb:.4f} MB")



model1: pruning 35.00% of filters per layer
  Convolution 0: 16 -> 10 filters
  Convolution 4: 32 -> 21 filters
  Convolution 8: 64 -> 42 filters
  Convolution 12: 128 -> 83 filters
  Output shape: (1, 10)
  Parameters: 118,170 -> 55,074 (53.4% reduction)
  Size: 0.4746 -> 0.2216 MB

model2: pruning 54.30% of filters per layer
  Convolution 0: 32 -> 15 filters
  Convolution 4: 64 -> 29 filters
  Convolution 8: 128 -> 58 filters
  Convolution 12: 256 -> 117 filters
  Output shape: (1, 10)
  Parameters: 429,866 -> 99,700 (76.8% reduction)
  Size: 1.7233 -> 0.4006 MB

model3: pruning 77.15% of filters per layer
  Convolution 0: 64 -> 15 filters
  Convolution 4: 128 -> 29 filters
  Convolution 8: 256 -> 58 filters
  Convolution 12: 512 -> 117 filters
  Output shape: (1, 10)
  Parameters: 1,633,866 -> 99,700 (93.9% reduction)
  Size: 6.5432 -> 0.4006 MB


In [4]:
# Evaluate the original and pruned models on the same validation split.
train_loader, val_loader, test_loader = create_data_loaders(batch_size=64, num_workers=12)
for name in models:
    for label, candidate in (("original", models[name]), ("pruned", pruned_models[name])):
        correct, total = evaluate_model(candidate, val_loader, device)
        print(f"{name} {label}: {correct}/{total} ({correct / total:.2%})")


model1 original: 2601/2700 (96.33%)
model1 pruned: 435/2700 (16.11%)
model2 original: 2621/2700 (97.07%)
model2 pruned: 440/2700 (16.30%)
model3 original: 2620/2700 (97.04%)
model3 pruned: 395/2700 (14.63%)


In [5]:
# Fine-tune each pruned model and save each best validation checkpoint separately.
output_dir = Path("../Checkpoints") / f"pruning_{uuid4().hex}"
output_dir.mkdir(parents=True, exist_ok=False)
histories = {}
for name, pruned in pruned_models.items():
    checkpoint = output_dir / f"{name}_pruned_best.pt"
    print(f"\nFine-tuning {name}; checkpoint: {checkpoint}")
    histories[name] = train_model(
        model=pruned, train_loader=train_loader, val_loader=val_loader,
        device=device, epochs=20, lr=0.0001, gamma=0.8,
        checkpoint_path=checkpoint,
    )
    pruned.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    correct, total = evaluate_model(pruned, val_loader, device)
    print(f"{name} fine-tuned validation accuracy: {correct}/{total} ({correct / total:.2%})")



Fine-tuning model1; checkpoint: ../Checkpoints/pruning_e396bc086299417cb7e469aef0592669/model1_pruned_best.pt
Training model...
Epoch 1, Loss: 2.1948, Accuracy: 0.5527, LR: 0.0001000000, Val loss: 0.9760, Val accuracy: 0.6874
Epoch 2, Loss: 0.8576, Accuracy: 0.7345, LR: 0.0001000000, Val loss: 0.6757, Val accuracy: 0.7859
Epoch 3, Loss: 0.6798, Accuracy: 0.7819, LR: 0.0001000000, Val loss: 0.5426, Val accuracy: 0.8344
Epoch 4, Loss: 0.5853, Accuracy: 0.8106, LR: 0.0000800000, Val loss: 0.4727, Val accuracy: 0.8530
Epoch 5, Loss: 0.5438, Accuracy: 0.8208, LR: 0.0000800000, Val loss: 0.4352, Val accuracy: 0.8615
Epoch 6, Loss: 0.4840, Accuracy: 0.8377, LR: 0.0000800000, Val loss: 0.4040, Val accuracy: 0.8730
Epoch 7, Loss: 0.4553, Accuracy: 0.8444, LR: 0.0000640000, Val loss: 0.3906, Val accuracy: 0.8733
Epoch 8, Loss: 0.4408, Accuracy: 0.8512, LR: 0.0000640000, Val loss: 0.3769, Val accuracy: 0.8785
Epoch 9, Loss: 0.4301, Accuracy: 0.8572, LR: 0.0000640000, Val loss: 0.3469, Val accura

In [6]:
# The pruned checkpoint contains weights for the smaller architecture.
# To load it later, first recreate the pruned architecture using prune_filters()
# on the matching original model with prune_ratios[name], then load the saved state dictionary.
print(f"Fine-tuned checkpoints saved in {output_dir}")


Fine-tuned checkpoints saved in ../Checkpoints/pruning_e396bc086299417cb7e469aef0592669
